# Modeling

## Import Dependencies

In [10]:
import os
import warnings

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, f1_score, log_loss
import optuna
import mlflow

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Data Loading

In [11]:
data = os.path.abspath(os.path.join(os.path.dirname(__name__), "..", "data"))
experiments = os.path.abspath(os.path.join(os.path.dirname(__name__), "..", "experiments"))

regression_path = os.path.join(data, "regression.csv")
classification_path = os.path.join(data, "classification.csv")

regression_dataset = pd.read_csv(filepath_or_buffer=regression_path)
classification_dataset = pd.read_csv(filepath_or_buffer=classification_path)

## Regression Models

#### Split dataset to inputs (x), target (y) for train and test

In [3]:
X = regression_dataset.drop("Price", axis=1)
y = regression_dataset["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

#### Compute metrics

1. Root Mean Squared Error
2. Mean Absolute Error
3. R2 Correlation

In [4]:
def regression_eval_metrics(actual, prediction) -> tuple[float, float, float]: #TODO
    """
    TODO
    :param actual: 
    :param prediction: 
    :return: 
    """
    rmse = mean_squared_error(actual, prediction, squared=False)
    mae = mean_absolute_error(actual, prediction)
    r2 = r2_score(actual, prediction)

    return round(float(rmse), 2), round(float(mae), 2), round(float(r2), 2)

#### Initialize models
1. Linear Regression
2. Lasso Regression (L1)
3. Ridge Regression (L2)
4. Random Forest Regression

In [5]:
regression_models = [("Linear Regression", LinearRegression(), (X_train, y_train), (X_test, y_test)),
                     ("Lasso Regression", Lasso(), (X_train, y_train), (X_test, y_test)),
                     ("Ridge Regression", Ridge(), (X_train, y_train), (X_test, y_test)),
                     ("Random Forest Regression", RandomForestRegressor(), (X_train, y_train), (X_test, y_test))]

#### Rounding predictions to the closest predicted price

In [6]:
def round_to_nearest(predicted_value, threshold=2.5): #TODO
    """
    TODO
    :param predicted_value: 
    :param threshold: 
    :return: 
    """
    lower_multiple = 5 * (predicted_value // 5)
    
    upper_multiple = lower_multiple + 5
    
    if predicted_value - lower_multiple < threshold:
        return lower_multiple
    
    else:
        return upper_multiple

#### Tuning the hyperparameters

In [7]:
def regression_objective(trial, model_name, model, X_train, y_train, X_test, y_test): #TODO
    """
    TODO
    :param trial: 
    :param model_name: 
    :param model: 
    :param X_train: 
    :param y_train: 
    :param X_test: 
    :param y_test: 
    :return: 
    """
    if model_name == "Lasso Regression":
        alpha = trial.suggest_loguniform("alpha", 0.1, 1)
        max_iter = trial.suggest_int("max_iter", 1000, 10000)

        model.set_params(alpha=alpha, max_iter=max_iter)

    elif model_name == "Ridge Regression":
        alpha = trial.suggest_loguniform("alpha", 0.1, 1)
        max_iter = trial.suggest_int("max_iter", 1000, 10000)

        model.set_params(alpha=alpha, max_iter=max_iter)

    elif model_name == "Random Forest Regression":
        n_estimators = trial.suggest_int("n_estimators", 100, 500)
        max_depth = trial.suggest_int("max_depth", 5, 30)

        model.set_params(n_estimators=n_estimators, max_depth=max_depth)

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    predictions_rounded = [round_to_nearest(prediction) for prediction in predictions]


    rmse, mae, r2 = regression_eval_metrics(y_test, predictions_rounded)

    return rmse

#### Training and evaluating the model performance, and logging the results and configurations

In [8]:
for model_name, model, train, test in regression_models:
    X_train, y_train = train
    X_test, y_test = test

    study = optuna.create_study(direction="minimize", study_name=model_name)
    study.optimize(lambda trial: regression_objective(trial, model_name, model, X_train, y_train, X_test, y_test), n_trials=50)

    best_params = study.best_params
    model.set_params(**best_params)

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    predictions_rounded = [round_to_nearest(prediction) for prediction in predictions]

    rmse, mae, r2 = regression_eval_metrics(y_test, predictions_rounded)

    print(f"Model: {model_name} >> RMSE: {rmse}, MAE: {mae}, R2: {r2}")

    path = os.path.join(experiments, model_name)
    mlflow.set_tracking_uri(path)

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(best_params)
        mlflow.log_metrics({"RMSE": rmse, "MAE": mae, "R2": r2})
        mlflow.sklearn.log_model(model, "model", input_example=X_test.iloc[[0]])

Model: Linear Regression >> RMSE: 8.26, MAE: 5.34, R2: 0.2
Model: Lasso Regression >> RMSE: 8.7, MAE: 5.6, R2: 0.11
Model: Ridge Regression >> RMSE: 8.21, MAE: 5.31, R2: 0.2
Model: Random Forest Regression >> RMSE: 7.98, MAE: 4.61, R2: 0.25


## Classification Models

In [12]:
X = classification_dataset.drop("Price Category", axis=1)
y = classification_dataset["Price Category"]

y = y.map({"Bronze": 0, "Silver": 1, "Gold": 2})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

In [27]:
def classification_eval_metrics(actual, prediction, probabilities) -> tuple[float, float, float]: #TODO
    """
    TODO probabilities
    :param actual: 
    :param prediction: 
    :param probabilities:
    :return: 
    """
    loss = log_loss(actual, probabilities)
    accuracy = accuracy_score(actual, prediction)
    f1 = f1_score(actual, prediction, average="macro")
    
    return round(float(loss), 2), round(float(accuracy), 2), round(float(f1), 2)

In [143]:
classification_models = [("Logistic Regression", LogisticRegression(), (X_train, y_train), (X_test, y_test)),
                         ("Random Forest Classifier", RandomForestClassifier(), (X_train, y_train), (X_test, y_test))]

In [7]:
def classification_objective(trial, model_name, model, X_train, y_train, X_test, y_test): #TODO
    """
    TODO
    :param trial: 
    :param model_name: 
    :param model: 
    :param X_train: 
    :param y_train: 
    :param X_test: 
    :param y_test: 
    :return: 
    """
    if model_name == "Logistic Regression":
        c = trial.suggest_loguniform("C", 0.1, 1)
        max_iter = trial.suggest_int("max_iter", 5000, 10000)

        model.set_params(multi_class="multinomial", solver="lbfgs", random_state=42, C=c, max_iter=max_iter)

    elif model_name == "Random Forest Classifier":
        class_weight = {0: 1, 1: 5, 2: 10}
        n_estimators = trial.suggest_int("n_estimators", 100, 500)
        max_depth = trial.suggest_int("max_depth", 5, 50)
    
        model.set_params(class_weight=class_weight, random_state=42, n_estimators=n_estimators, max_depth=max_depth)


    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)
    
    loss, accuracy, f1 = classification_eval_metrics(y_test, predictions, probabilities)

    return f1

In [8]:
for model_name, model, train, test in classification_models:
    X_train, y_train = train
    X_test, y_test = test

    study = optuna.create_study(direction="maximize", study_name=model_name)
    study.optimize(lambda trial: classification_objective(trial, model_name, model, X_train, y_train, X_test, y_test), n_trials=50)

    best_params = study.best_params
    model.set_params(**best_params)

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)

    loss, accuracy, f1 = classification_eval_metrics(y_test, predictions, probabilities)

    print(f"Model: {model_name} >> Loss: {loss}, Accuracy: {accuracy}, F1 Score: {f1}")

    path = os.path.join(experiments, model_name)
    mlflow.set_tracking_uri(path)

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(best_params)
        mlflow.log_metrics({"Loss": loss, "Accuracy": accuracy, "F1 Score": f1})
        mlflow.sklearn.log_model(model, "model", input_example=X_test.iloc[[0]])

Model: Logistic Regression >> Loss: 0.35, Accuracy: 0.87, F1 Score: 0.48
Model: Random Forest Classifier >> Loss: 0.59, Accuracy: 0.87, F1 Score: 0.55
